# 01 — Download ACS PUMS Data for Washington State

This notebook downloads American Community Survey (ACS) Public Use Microdata Sample (PUMS)
person-level files for Washington state from the Census Bureau.

**Data source:** [Census Bureau ACS PUMS](https://www.census.gov/programs-surveys/acs/microdata.html)

**Years:** 2013–2019, 2021–2023 (skipping 2020 experimental estimates)

**PUMA vintages:**
- 2010-based PUMAs (used in 2013–2021 ACS)
- 2020-based PUMAs (used in 2022+ ACS)

In [ ]:
import os
import requests
import zipfile
import io
import pandas as pd
from tqdm.auto import tqdm

DATA_DIR = os.path.abspath(os.path.join('..', 'data'))
RAW_DIR = os.path.join(DATA_DIR, 'raw', 'acs_pums')
DERIVED_DIR = os.path.join(DATA_DIR, 'derived')
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(DERIVED_DIR, exist_ok=True)

# Years to download (skip 2020 — experimental due to COVID low response)
YEARS = list(range(2013, 2020)) + list(range(2021, 2024))
print(f'Will download ACS 1-Year PUMS for WA state: {YEARS}')

## Download PUMA reference data

The 2010-based PUMA names come from the Census Bureau gazetteer file.
The 2020-based PUMA names are hardcoded from the Census Bureau's
[2020 PUMA Names PDF](https://www2.census.gov/geo/pdfs/reference/puma2020/2020_PUMA_Names.pdf).

In [ ]:
# Download 2010 PUMA gazetteer (has PUMA codes + names)
gaz_dest = os.path.join(RAW_DIR, '2010_Gaz_PUMAs_national.txt')
if not os.path.exists(gaz_dest):
    url = 'https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2010_Gaz_PUMAs_national.zip'
    print(f'Downloading 2010 PUMA gazetteer...')
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
        with zf.open('2010_Gaz_PUMAs_national.txt') as src:
            with open(gaz_dest, 'wb') as dst:
                dst.write(src.read())
    print(f'  saved → {gaz_dest}')
else:
    print('2010 PUMA gazetteer: cached')

In [ ]:
# Parse 2010 PUMAs for Washington state
gaz = pd.read_csv(gaz_dest, sep='\t', dtype=str, encoding='latin-1')
gaz.columns = gaz.columns.str.strip()
wa_2010 = gaz[gaz['USPS'] == 'WA'].copy()
# GEOID = state_fips (2) + puma_code (5)
wa_2010['puma'] = wa_2010['GEOID'].str[2:].astype(int)
wa_2010['puma_name'] = wa_2010['NAME']

seattle_2010 = wa_2010[wa_2010['puma_name'].str.contains('Seattle', case=False, na=False)]
print('2010-based Seattle PUMAs (used in 2013–2021 ACS):')
for _, row in seattle_2010.iterrows():
    print(f"  {row['puma']:>5d}  {row['puma_name']}")

print()

# 2020-based Seattle PUMAs (from Census 2020 PUMA Names PDF)
seattle_pumas_2020 = {
    23312: 'Seattle City (West Seattle-Industrial)',
    23313: 'Seattle City (Southeast)',
    23314: 'Seattle City (Central)',
    23315: 'Seattle City (Lake Union-Downtown)',
    23316: 'Seattle City (Northwest)',
    23317: 'Seattle City (Northeast)',
    23318: 'Seattle City (North)',
}
print('2020-based Seattle PUMAs (used in 2022+ ACS):')
for code, name in sorted(seattle_pumas_2020.items()):
    print(f'  {code:>5d}  {name}')

## Download ACS PUMS person files

Download the person-level PUMS CSV for Washington state from the Census FTP site.
We extract only the columns needed for commute analysis and save as parquet.

In [ ]:
# Columns to extract from each year's PUMS file
TARGET_COLS_UPPER = {
    'SERIALNO', 'ST', 'PUMA', 'PUMA00', 'PUMA10', 'PUMA20',
    'POWSP', 'POWPUMA',
    'JWTR', 'JWTRNS', 'JWRIP', 'DRIVESP',
    'JWMNP', 'JWAP', 'JWDP',
    'PWGTP', 'ESR',
}
# Add replicate weights for standard error computation
for i in range(1, 81):
    TARGET_COLS_UPPER.add(f'PWGTP{i}')


def col_filter(col_name):
    """Filter function for usecols — keep only target columns."""
    return col_name.upper() in TARGET_COLS_UPPER


print(f'Will extract up to {len(TARGET_COLS_UPPER)} columns from each file')

In [ ]:
for year in tqdm(YEARS, desc='ACS PUMS download'):
    outfile = os.path.join(DERIVED_DIR, f'acs_pums_wa_commute_{year}.parquet')
    if os.path.exists(outfile):
        print(f'{year}: cached')
        continue

    url = f'https://www2.census.gov/programs-surveys/acs/data/pums/{year}/1-Year/csv_pwa.zip'
    print(f'{year}: downloading...')

    r = requests.get(url, timeout=600)
    r.raise_for_status()

    content = io.BytesIO(r.content)
    with zipfile.ZipFile(content) as zf:
        csv_names = [n for n in zf.namelist() if n.lower().endswith('.csv')]
        if not csv_names:
            print(f'{year}: ERROR — no CSV found in zip')
            continue

        print(f'{year}: reading {csv_names[0]}...')
        with zf.open(csv_names[0]) as f:
            df = pd.read_csv(f, usecols=col_filter, low_memory=False)

    # Standardize column names to uppercase
    df.columns = df.columns.str.upper()
    df['YEAR'] = year

    df.to_parquet(outfile, index=False)
    print(f'{year}: saved {len(df):,} records')

## Summary

In [ ]:
print('Downloaded data summary')
print('=' * 65)
total = 0
for year in YEARS:
    f = os.path.join(DERIVED_DIR, f'acs_pums_wa_commute_{year}.parquet')
    if os.path.exists(f):
        df = pd.read_parquet(f)
        n = len(df)
        total += n
        commute_cols = [c for c in df.columns if c in ('JWTR', 'JWTRNS')]
        rip_cols = [c for c in df.columns if c in ('JWRIP', 'DRIVESP')]
        print(f'  {year}:  {n:>8,} records   mode_var={commute_cols}  rip_var={rip_cols}')
    else:
        print(f'  {year}:  NOT DOWNLOADED')
print(f'{"":>6s}  {total:>8,} total records')